# 03 · Analyze results
Reads the prediction files in `RUNS_BASE` (no GPU, no torch needed): accuracy tables across select scores and modes,
and paired McNemar tests of each steered setting against the unsteered baseline (alpha = 0).

In [ ]:
# Storage locations: set them BEFORE importing caga (see caga/paths.py).
# CAGA_BASE defaults to ~/caga; point it at a disk with ~60 GB free.
import os, sys
# os.environ["CAGA_BASE"] = "/path/to/large/disk/caga"

# Make the repository importable whether the notebook runs from the repo root or from notebooks/.
for cand in (os.getcwd(), os.path.dirname(os.getcwd())):
    if os.path.isdir(os.path.join(cand, "caga")):
        if cand not in sys.path:
            sys.path.insert(0, cand)
        break

In [ ]:
import pandas as pd
from caga import config
from caga.results import collect, paired_test

DATASET = "mmar"
df = collect(DATASET)
df

### One mode, all select scores

In [ ]:
MODE = "final"
df[df["mode"] == MODE]

### Paired significance of the best alpha
`best_alpha` is chosen on the evaluation set itself, so treat it as an upper bound; p-values below are not corrected for that choice or for multiple comparisons.

In [ ]:
rows = []
for _, r in df.dropna(subset=["best_alpha"]).iterrows():
    t = paired_test(DATASET, r["mode"], r["K"], float(r["best_alpha"]),
                    select_score=r["select_score"], pertoken_topk=r["pertoken_topk"])
    rows.append({"select_score": r["select_score"], "mode": r["mode"], "K": r["K"],
                 "alpha": r["best_alpha"], **t})
pd.DataFrame(rows).sort_values("p_value")